In [70]:
%pip install -q cleanco

import re
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cleanco import basename
from matplotlib.ticker import PercentFormatter
from scipy import stats
from sqlalchemy import func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN, TLS202_APPLN_TITLE, TLS203_APPLN_ABSTR,
    TLS206_PERSON, TLS207_PERS_APPLN, TLS224_APPLN_CPC,
)


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [71]:
ENV = "TEST"
NAME_METHOD = "dictionary"

SECTOR_NAME = "Surface drones (unmanned surface vessels)"
SECTOR_CPC_PREFIXES = (
    "B63B2035/006",   # unmanned surface vessels, e.g. remotely controlled
    "B63B2035/007",   # ... autonomously operating
    "B63B2035/008",   # ... remotely controlled
)
SECTOR_KEYWORDS = (
    "unmanned surface vessel", "unmanned surface vehicle",
    "autonomous surface vessel", "autonomous surface vehicle",
    "uncrewed surface", "surface drone", "drone boat",
    "unmanned boat", "autonomous boat", "unmanned ship", "autonomous ship",
    "unmanned vessel", "robotic boat",
)

YEAR_MIN, YEAR_MAX = 2000, 2025
LAST_COMPLETE_YEAR = YEAR_MAX - 3
IPR_TYPE = "PI"
SPLIT_YEAR = 2015
WINDOW = 3
TOP_N = 15

patstat = PatstatClient(env=ENV)
db = patstat.orm()


## Sector definition: families found by CPC codes and by keywords

In [72]:
def code_filter(column, codes):
    clean = func.replace(column, " ", "")
    return or_(*[clean.like(f"{c}%") for c in codes])

def keyword_filter(terms):
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(*[title.like(f"%{t}%") for t in terms], *[abstract.like(f"%{t}%") for t in terms])

family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")
q_cpc = (db.query(family_id)
         .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
         .filter(code_filter(TLS224_APPLN_CPC.cpc_class_symbol, SECTOR_CPC_PREFIXES)))
q_kw = (db.query(family_id)
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_filter(SECTOR_KEYWORDS)))

by_code = set(patstat.df(q_cpc)["docdb_family_id"])
by_kw = set(patstat.df(q_kw)["docdb_family_id"])
print("by codes:", len(by_code), "| by keywords:", len(by_kw), "| total:", len(by_code | by_kw))


by codes: 127 | by keywords: 125 | total: 199


## Quality check of the sector

In [73]:
sector_families = by_code | by_kw

q_titles = (db.query(TLS201_APPLN.docdb_family_id, TLS202_APPLN_TITLE.appln_title)
            .join(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
            .filter(TLS201_APPLN.docdb_family_id.in_(list(sector_families))))
titles = patstat.df(q_titles).drop_duplicates("docdb_family_id").set_index("docdb_family_id")["appln_title"]

pd.set_option("display.max_colwidth", 120)
for label, ids in [("codes only", by_code - by_kw), ("keywords only", by_kw - by_code)]:
    print(f"\n=== {label}: 10 random titles ===")
    sample = titles[titles.index.isin(ids)].sample(10, random_state=1)
    for t in sample:
        print("-", t)



=== codes only: 10 random titles ===
- PROCESS FOR HARVESTING, STORING, AND USING RENEWABLE ENERGY TO PROPEL AND POWER BOATS AND SHIPS, AND MAXIMIZE THEIR AVERAGE SPEED
- Buoy releasing device for hydrological monitoring
- Method and system for a towed vessel suitable for transporting liquids
- Intelligent collection and treatment system and method for water hyacinth in water area
- Photovoltaic hybrid power small unmanned scientific investigation ship
- Self-powered computing buoy
- Modular autonomous surface robot of catamaran type
- BATEAU DE SURFACE AUTONOME AMELIORE
- Energy self-contained oceanic drone for AI-based marine information survey and surveillance and method using same
- Installation flottante de production et de stockage d’hydrogène hors réseau électrique terrestre

=== keywords only: 10 random titles ===
- A compound-powered unmanned craft
- Sea-air-underwater integrated self-supply detection device and detection method thereof
- Be used for clear solar energy unmann

In [74]:
code_only = list(by_code - by_kw)

q_text = (db.query(TLS201_APPLN.docdb_family_id, TLS202_APPLN_TITLE.appln_title, TLS203_APPLN_ABSTR.appln_abstract)
          .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
          .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
          .filter(TLS201_APPLN.docdb_family_id.in_(code_only)))
text = patstat.df(q_text).fillna("")
text["text"] = (text["appln_title"] + " " + text["appln_abstract"]).str.lower()

# one long text per family (all its applications together)
family_text = text.groupby("docdb_family_id")["text"].apply(" ".join)

DRONE_WORDS = r"unmanned|uncrewed|autonom|drone|remote.?control|\busv\b|unattended|robot"
mentions = family_text.str.contains(DRONE_WORDS, regex=True)

kept, dropped = set(mentions[mentions].index), set(mentions[~mentions].index)
print("codes-only families:", len(code_only), "| kept:", len(kept), "| dropped:", len(dropped))

print("\n=== 10 dropped titles (check that they are really noise) ===")
for t in titles[titles.index.isin(dropped)].head(10):
    print("-", t)


codes-only families: 74 | kept: 48 | dropped: 26

=== 10 dropped titles (check that they are really noise) ===
- Wave power
- Wave-powered devices configured for nesting
- Method and system for a towed vessel suitable for transporting liquids
- Docking facility for tower on sea and Tower on sea having the same
- Catamaran surface vessel with removable mission-specific payload module
- Submersible vessel having retractable wing and keel assemblies
- Adaptable modular power system (AMPS)
- PROCESS FOR HARVESTING, STORING, AND USING RENEWABLE ENERGY TO PROPEL AND POWER BOATS AND SHIPS, AND MAXIMIZE THEIR AVERAGE SPEED
- Rainwater harvesting system
- AN AQUATIC MANAGEMENT SYSTEM


### Final fix: two more drone words, and the final sector

In [75]:
DRONE_WORDS = r"unmanned|uncrewed|autonom|drone|remote.?control|\busv\b|unattended|robot|payload|glider"
mentions = family_text.str.contains(DRONE_WORDS, regex=True)
kept = set(mentions[mentions].index)
print("codes-only kept:", len(kept), "of", len(code_only))

# FINAL SECTOR: keywords OR (codes AND drone words)
sector_families = by_kw | kept
print("final sector:", len(sector_families), "families")


codes-only kept: 50 of 74
final sector: 175 families


## Applicants and CPC codes of the sector families

In [76]:
# 1. sector families with their year (= earliest priority year), real patents only
family_year = func.min(TLS201_APPLN.earliest_filing_year)
families = (
    db.query(TLS201_APPLN.docdb_family_id.label("unit_id"), family_year.label("year"))
    .filter(TLS201_APPLN.ipr_type == IPR_TYPE,
            TLS201_APPLN.appln_id < 900000000,                # drop PATSTAT's internal dummy rows
            TLS201_APPLN.docdb_family_id.in_(list(sector_families)))
    .group_by(TLS201_APPLN.docdb_family_id)
    .having(family_year.between(YEAR_MIN, YEAR_MAX))
    .subquery()
)

# 2. the applicants of every family: one row per (family, person)
q_sector = (
    db.query(families.c.unit_id, families.c.year,
             TLS206_PERSON.psn_id, TLS206_PERSON.psn_name, TLS206_PERSON.han_name,
             TLS206_PERSON.psn_sector, func.trim(TLS206_PERSON.person_ctry_code).label("ctry"))
    .select_from(families)
    .join(TLS201_APPLN, TLS201_APPLN.docdb_family_id == families.c.unit_id)
    .join(TLS207_PERS_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id)
    .join(TLS206_PERSON, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id)
    .filter(TLS207_PERS_APPLN.applt_seq_nr > 0)          # applicants only, not inventors
    .distinct()
)
raw_sector = patstat.df(q_sector)

# 3. the CPC codes of every family (needed later for the subfields)
q_cpc_fam = (
    db.query(families.c.unit_id, families.c.year,
             func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "").label("cpc"))
    .select_from(families)
    .join(TLS201_APPLN, TLS201_APPLN.docdb_family_id == families.c.unit_id)
    .join(TLS224_APPLN_CPC, TLS224_APPLN_CPC.appln_id == TLS201_APPLN.appln_id)
    .distinct()
)
family_cpc = patstat.df(q_cpc_fam)

print(raw_sector["unit_id"].nunique(), "families with applicants |", len(raw_sector), "rows")
raw_sector.head()

153 families with applicants | 198 rows


,unit_id,year,psn_id,psn_name,han_name,psn_sector,ctry
0,8849607,2000,8294660,ETAT FRANCAIS REPRESENTE PAR LE DELEGUE GENERAL POUR L'ARMEMENT,ETAT FR REPRESENTE PAR LE DELEGUE GENAL POUR L ARMEMENT,GOV NON-PROFIT,FR
1,34117148,2003,30570259,SOLAR SAILOR,SOLAR SAILOR PTY LTD,COMPANY,AU
2,34117148,2003,6096400,"DANE, ROBERT, A","Dane, Robert A.",INDIVIDUAL,AU
3,34117148,2003,16992982,"KILBOURN, EDWARD, PAYNE","Kilbourn, Edward Payne",INDIVIDUAL,US
4,37252408,2003,17391575,KMPRC( KOREA MARINE POLLUTION RESPONSE CORP.),KMPRC KR MARINE POLLUTION RESPONSE CORP,COMPANY,KR


## Raw applicant names: how many spellings?

In [77]:
print("different psn_name:", raw_sector["psn_name"].nunique())
print("different han_name:", raw_sector["han_name"].nunique())
print("different psn_id:  ", raw_sector["psn_id"].nunique())

(raw_sector.groupby("psn_name")["unit_id"].nunique()
           .sort_values(ascending=False).head(20))


different psn_name: 163
different han_name: 165
different psn_id:   167


psn_name
SHANGHAI JIAO TONG UNIVERSITY                             5
HARBIN ENGINEERING UNIVERSITY                             4
NANJING UNIVERSITY OF INFORMATION SCIENCE & TECHNOLOGY    3
ZHEJIANG UNIVERSITY                                       3
SHANGHAI OCEAN UNIVERSITY                                 3
SHANGHAI UNIVERSITY                                       3
ZHEJIANG OCEAN UNIVERSITY                                 3
KOREA INSTITUTE OF OCEAN SCIENCE & TECHNOLOGY             3
SOUTH CHINA UNIVERSITY OF TECHNOLOGY                      2
SOLAR SAILOR                                              2
LIQUID ROBOTICS                                           2
HOLEMANS, WALTER                                          2
HEBEI UNIVERSITY OF TECHNOLOGY                            2
OCEAN UNIVERSITY OF CHINA                                 2
JIANGSU UNIVERSITY OF SCIENCE AND TECHNOLOGY              2
GUANGZHOU SHUNHAI SHIPYARDS                               2
GEOSYSTEM RESEARCH CORPORATION 

## Name cleaning: methods A (dictionary) and B (cleanco)

In [78]:
LEGAL_FORMS = {
    "INC", "INCORPORATED", "CORP", "CORPORATION", "CO", "COMPANY", "LTD", "LIMITED", "LLC", "LP", "PLC",
    "GMBH", "AG", "KG", "MBH", "SE", "SA", "SAS", "SPA", "SRL", "BV", "NV", "AB", "AS", "ASA", "OY", "OYJ",
    "KK", "KABUSHIKI", "KAISHA", "LTDA", "PTY", "THE", "AKTIENGESELLSCHAFT", "GESELLSCHAFT",
}

def clean_with_dictionary(name):
    "Method A: upper-case, keep letters/digits, drop legal-form words at the start and end."
    words = re.sub(r"[^A-Z0-9 ]", " ", str(name).upper()).split()
    while words and words[-1] in LEGAL_FORMS:
        words.pop()
    while words and words[0] in LEGAL_FORMS:
        words.pop(0)
    return " ".join(words)

def clean_with_cleanco(name):
    "Method B: cleanco removes legal forms first, then the same cleaning as A."
    return clean_with_dictionary(basename(str(name)))

# how many different names does each method leave?
names = raw_sector["psn_name"].drop_duplicates()
print("psn_name as is:", names.nunique())
print("method A:      ", names.map(clean_with_dictionary).nunique())
print("method B:      ", names.map(clean_with_cleanco).nunique())

# which names did method A merge?
merged = pd.DataFrame({"psn_name": names, "A": names.map(clean_with_dictionary)})
merged[merged.duplicated("A", keep=False)].sort_values("A")

psn_name as is: 163
method A:       160
method B:       160


,psn_name,A
69,CAS (HEFEI) INSTITUTE OF TECHNOLOGY INNOVATION,CAS HEFEI INSTITUTE OF TECHNOLOGY INNOVATION
70,CAS HEFEI INSTITUTE OF TECHNOLOGY INNOVATION,CAS HEFEI INSTITUTE OF TECHNOLOGY INNOVATION
60,ETME: PEPPAS KAI SYNERGATES E.E.,ETME PEPPAS KAI SYNERGATES E E
62,Etme: Peppas Kai Synergates E.E.,ETME PEPPAS KAI SYNERGATES E E
123,THAYERMAHAN,THAYERMAHAN
124,"THAYERMAHAN, INC.",THAYERMAHAN


In [79]:

GOLD_STANDARD_FILE = "https://raw.githubusercontent.com/goksuyegen/innovation-management-datascience26/main/tutorial_1/PATSTAT_500_label_datav4.csv"
gold = pd.read_csv(GOLD_STANDARD_FILE).rename(columns={"comp": "name", "Group": "gold_group"})
print(f"{len(gold):,} names in {gold['gold_group'].nunique():,} organisations")


def pairs_inside(group_sizes):
    "Number of pairs inside groups of these sizes: n(n-1)/2 each."
    return sum(comb(int(n), 2) for n in group_sizes)


def precision_recall_f1(gold_groups, predicted_groups):
    df = pd.DataFrame({"gold": list(gold_groups), "pred": list(predicted_groups)})
    tp = pairs_inside(df.groupby(["gold", "pred"]).size())   # pairs we merged AND that are truly the same
    tp_fp = pairs_inside(df.groupby("pred").size())          # all pairs we merged
    tp_fn = pairs_inside(df.groupby("gold").size())          # all pairs that truly belong together
    precision = tp / tp_fp if tp_fp else 1.0
    recall = tp / tp_fn
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"groups found": df["pred"].nunique(), "precision": precision, "recall": recall, "F1": f1}


methods = {
    "no cleaning": lambda n: n,
    "A dictionary": clean_with_dictionary,
    "B cleanco": clean_with_cleanco,
}
pd.DataFrame({label: precision_recall_f1(gold["gold_group"], gold["name"].map(f))
              for label, f in methods.items()}).T.round(3)


3,192 names in 481 organisations


,groups found,precision,recall,F1
no cleaning,3192.0,1.000,0.000,0.000
A dictionary,3010.0,0.995,0.002,0.004
B cleanco,2984.0,0.996,0.003,0.005


In [80]:
def method_c(name):
    "Method C: the first word of the cleaned name."
    words = clean_with_dictionary(name).split()
    return words[0] if words else ""

methods["C first word"] = method_c
pd.DataFrame({label: precision_recall_f1(gold["gold_group"], gold["name"].map(f))
              for label, f in methods.items()}).T.round(3)


,groups found,precision,recall,F1
no cleaning,3192.0,1.000,0.000,0.000
A dictionary,3010.0,0.995,0.002,0.004
B cleanco,2984.0,0.996,0.003,0.005
C first word,536.0,0.961,0.923,0.942


In [81]:
check = pd.DataFrame({
    "psn_name": names,
    "A": names.map(clean_with_dictionary),
    "C": names.map(method_c),
})
over_merged = check.groupby("C").filter(lambda g: g["A"].nunique() > 1)
print("names merged by C but not by A:", len(over_merged))
over_merged.sort_values("C")


names merged by C but not by A: 67


,psn_name,A,C
109,CHANGSHA UNIVERSITY OF SCIENCE & TECHNOLOGY,CHANGSHA UNIVERSITY OF SCIENCE TECHNOLOGY,CHANGSHA
78,CHANGSHA NINGHU MACHINERY EQUIPMENT COMPANY,CHANGSHA NINGHU MACHINERY EQUIPMENT,CHANGSHA
147,CHINA SHIPBUILDING INDUSTRY CORPORATION NO.707 RESEARCH INSTITUTE,CHINA SHIPBUILDING INDUSTRY CORPORATION NO 707 RESEARCH INSTITUTE,CHINA
34,CHINA UNIVERSITY OF GEOSCIENCES (WUHAN),CHINA UNIVERSITY OF GEOSCIENCES WUHAN,CHINA
86,CHINESE ACADEMY OF SCIENCES,CHINESE ACADEMY OF SCIENCES,CHINESE
...,...,...,...
105,ZHEJIANG UNIVERSITY,ZHEJIANG UNIVERSITY,ZHEJIANG
91,ZHEJIANG OCEAN UNIVERSITY,ZHEJIANG OCEAN UNIVERSITY,ZHEJIANG
183,ZHEJIANG INSTITUTE OF HYDRAULICS & ESTUARY (ZHEJIANG MARINE PLANNING AND DESIGN INSTITUTE),ZHEJIANG INSTITUTE OF HYDRAULICS ESTUARY ZHEJIANG MARINE PLANNING AND DESIGN INSTITUTE,ZHEJIANG
162,ZHEJIANG YIMING TECHNOLOGY COMPANY,ZHEJIANG YIMING TECHNOLOGY,ZHEJIANG


## Applicant table: cleaned name (method A) and type

Method C scored best on the gold standard (F1 = 0.94) but on our sector it merges 67 names of unrelated organisations (e.g. all "ZHEJIANG …" universities and firms), because Chinese names start with a city. **We use method A.**

In [82]:
NAME_MERGES = {                        # checked by hand: cleaned name -> group name
    "GUANGZHOU SHUNHAI SHIP MAKING": "GUANGZHOU SHUNHAI SHIPYARDS",
    # CSIC research institutes -> one group (your decision!)
    # "CHINA SHIPBUILDING INDUSTRY CORPORATION NO 707 RESEARCH INSTITUTE": "CHINA SHIPBUILDING INDUSTRY CORPORATION (CSIC)",
    # "715TH RESEARCH INSTITUTE OF CHINA SHIPBUILDING INDUSTRY": "CHINA SHIPBUILDING INDUSTRY CORPORATION (CSIC)",
}



In [83]:
def applicant_type(psn_sector):
    "Turn PATSTAT's psn_sector into a few simple types."
    s = str(psn_sector).upper()
    if "UNIVERSITY" in s:
        return "university"
    if "COMPANY" in s:
        return "firm"
    if "GOV" in s:
        return "government / non-profit"
    if "INDIVIDUAL" in s:
        return "individual"
    return "unknown"

links = raw_sector.copy()
links["applicant"] = links["psn_name"].map(clean_with_dictionary).replace(NAME_MERGES)
links["type"] = links["psn_sector"].map(applicant_type)

print(links["type"].value_counts())
links.groupby("applicant")["unit_id"].nunique().sort_values(ascending=False).head(20)


type
firm                       69
university                 62
individual                 29
unknown                    25
government / non-profit    13
Name: count, dtype: int64


applicant
SHANGHAI JIAO TONG UNIVERSITY                           5
HARBIN ENGINEERING UNIVERSITY                           4
SHANGHAI OCEAN UNIVERSITY                               3
KOREA INSTITUTE OF OCEAN SCIENCE TECHNOLOGY             3
NANJING UNIVERSITY OF INFORMATION SCIENCE TECHNOLOGY    3
SHANGHAI UNIVERSITY                                     3
ZHEJIANG OCEAN UNIVERSITY                               3
ZHEJIANG UNIVERSITY                                     3
DALIAN MARITIME UNIVERSITY                              2
HOLEMANS WALTER                                         2
JIANGSU UNIVERSITY OF SCIENCE AND TECHNOLOGY            2
GEOSYSTEM RESEARCH                                      2
SOUTH CHINA UNIVERSITY OF TECHNOLOGY                    2
SOLAR SAILOR                                            2
LIQUID ROBOTICS                                         2
GUANGZHOU SHUNHAI SHIPYARDS                             2
HEBEI UNIVERSITY OF TECHNOLOGY                          2
OCEA

In [84]:
display(links.drop_duplicates("applicant")["type"].value_counts())
links.loc[links["type"] == "unknown", "psn_name"].unique()

type
firm                       64
university                 37
individual                 28
unknown                    19
government / non-profit    11
Name: count, dtype: int64

array(['ETME: PEPPAS KAI SYNERGATES E.E.',
       'Etme: Peppas Kai Synergates E.E.', 'E-4S',
       'CAS (HEFEI) INSTITUTE OF TECHNOLOGY INNOVATION',
       'SIGNAL VENTURES LTD', 'THAYERMAHAN, INC.',
       'SOLAR SPLASH CO., LTD.', 'Xunord S.r.l.',
       'GUANGZHOU XIAONIAN INTELLIGENT TECHNOLOGY CO., LTD.',
       'SHANGHAI TRAFFIC UNIVERSITY',
       'JIANGSU HUBYLIN ENVIRONMENTAL PROTECTION EQUIPMENT LIMITED COMPANY',
       'WUHAN TECHNOLOGY UNIVERSITY',
       'NANTONG SCIENCE AND ENGINEERING COLLEGE',
       'ZHEJIANG UNIVERSITY RESEARCH INSTITUTE IN HAINAN',
       'SHANDONG OLD NETWORK ELECTRONIC SCIENCE AND TECHNOLOGY CO., LTD',
       'SHANDONG LABOR OCCUPATIONAL TECHNOLOGY COLLEGE (SHANDONG LABOR TECHNICIAN COLLEGE)',
       'ZHEJIANG INSTITUTE OF HYDRAULICS & ESTUARY (ZHEJIANG MARINE PLANNING AND DESIGN INSTITUTE)',
       'WUHAN HUAI MU ARMY TECHNOLOGY CO., LTD.',
       'HUBEI WATERWAY ENGINEERING CO., LTD.',
       'ZIBO WATER CONSERVANCY SURVEY AND DESIGN INSTITUTE 

In [85]:
FIRM_WORDS = {"CO", "LTD", "LIMITED", "INC", "COMPANY", "CORP", "LLC", "GMBH", "SRL"}

def type_from_name(name):
    "Guess the type from the words of the name (used only when psn_sector says nothing)."
    words = set(re.sub(r"[^A-Z0-9 ]", " ", str(name).upper()).split())
    if words & FIRM_WORDS:
        return "firm"
    if words & {"UNIVERSITY", "COLLEGE"}:
        return "university"
    if words & {"INSTITUTE", "ACADEMY"}:
        return "government / non-profit"
    return "unknown"

unknown = links["type"] == "unknown"
links.loc[unknown, "type"] = links.loc[unknown, "psn_name"].map(type_from_name)

TYPE_FIXES = {                       # checked by hand
    "XUNORD S R L": "firm",
    "ETME PEPPAS KAI SYNERGATES E E": "firm",
}
links["type"] = links["applicant"].map(TYPE_FIXES).fillna(links["type"])

links.loc[links["type"] == "unknown", "psn_name"].unique()   # what is still unknown?

array(['E-4S'], dtype=object)

## Country of each applicant

In [86]:
links["ctry"] = links["ctry"].replace("", np.nan)

per_applicant = links.groupby("applicant")["ctry"].apply(lambda c: c.notna().any())
print("applicants with no country at all:", (~per_applicant).sum(), "of", len(per_applicant))

applicants with no country at all: 96 of 159


Get the office of each family 

In [87]:
q_offices = (db.query(families.c.unit_id, TLS201_APPLN.appln_auth)
             .select_from(families)
             .join(TLS201_APPLN, TLS201_APPLN.docdb_family_id == families.c.unit_id)
             .distinct())
family_offices = patstat.df(q_offices)
family_offices["appln_auth"].value_counts().head(10)


appln_auth
CN    105
US     28
KR     14
WO     12
EP      5
AU      2
GB      1
MA      1
DK      1
PL      1
Name: count, dtype: int64

one country per applicant 4b

In [88]:
def most_frequent(values):
    return values.value_counts().index[0]

# 1. the country written on the patents (most frequent one)
known = links.dropna(subset=["ctry"]).drop_duplicates(["unit_id", "applicant", "ctry"])
main_country = known.groupby("applicant")["ctry"].agg(most_frequent)

# 2. fallback: the most frequent national office of its families
national = family_offices[~family_offices["appln_auth"].isin(["EP", "WO"])]
offices = links[["unit_id", "applicant"]].drop_duplicates().merge(national, on="unit_id")
main_office = offices.groupby("applicant")["appln_auth"].agg(most_frequent)

# 3. combine: written country first, office if there is none
links["country"] = links["applicant"].map(main_country).fillna(links["applicant"].map(main_office))
links["country_source"] = np.where(links["applicant"].isin(main_country.index), "written",
                          np.where(links["country"].notna(), "office", "missing"))

# 4. hand fixes (checked by hand)
COUNTRY_FIXES = {"PEPPAS ANTONIOS": "GR"}
fix = links["applicant"].isin(COUNTRY_FIXES)
links.loc[fix, "country"] = links.loc[fix, "applicant"].map(COUNTRY_FIXES)
links.loc[fix, "country_source"] = "hand"

# one row per applicant, AFTER all the fixes
per_app = links.drop_duplicates("applicant")
display(per_app["country_source"].value_counts())
per_app["country"].value_counts().head(10)

country_source
office     95
written    63
hand        1
Name: count, dtype: int64

country
CN    97
US    33
KR    16
GB     4
AU     2
FR     2
GR     2
IN     1
NO     1
IT     1
Name: count, dtype: int64

In [89]:
pd.crosstab(per_app["country"], per_app["country_source"])


country_source,hand,office,written
country,,,
AU,0,0,2
CN,0,94,3
FR,0,0,2
GB,0,0,4
GR,1,0,1
IN,0,0,1
IT,0,0,1
KR,0,0,16
NO,0,0,1


In [90]:
per_app.loc[(per_app["country_source"] == "office") & (per_app["country"] != "CN"), ["applicant", "country"]]


,applicant,country
7,J3S,US


In [91]:
for word in ["SHUNHAI", "SHIPBUILDING", "PEPPAS"]:
    print(word, "->", sorted(links.loc[links["applicant"].str.contains(word), "applicant"].unique()))


SHUNHAI -> ['GUANGZHOU SHUNHAI SHIPYARDS']
SHIPBUILDING -> ['715TH RESEARCH INSTITUTE OF CHINA SHIPBUILDING INDUSTRY', 'CHINA SHIPBUILDING INDUSTRY CORPORATION NO 707 RESEARCH INSTITUTE']
PEPPAS -> ['ETME PEPPAS KAI SYNERGATES E E', 'PEPPAS ANTONIOS']


In [93]:
# one row per (family, applicant)
links = links.drop_duplicates(["unit_id", "applicant"])

# Q1 input: families per applicant, complete years only
complete = links[links["year"] <= LAST_COMPLETE_YEAR]
families_per_applicant = complete.groupby("applicant")["unit_id"].nunique()

print(len(families_per_applicant), "applicants,", families_per_applicant.sum(), "applicant-family pairs")
families_per_applicant.value_counts().sort_index()


156 applicants, 184 applicant-family pairs


unit_id
1    138
2     11
3      5
4      1
5      1
Name: count, dtype: int64

## Summary of the results so far (keep this cell at the end)

In [92]:
per_app = links.drop_duplicates("applicant")

print(f"ENV: {ENV}  |  sector: {SECTOR_NAME}")
print(f"\nSector families: {len(sector_families)} (codes {len(by_code)}, keywords {len(by_kw)}, codes kept after drone words {len(kept)})")
print(f"Families with applicants ({YEAR_MIN}-{YEAR_MAX}, {IPR_TYPE}): {raw_sector['unit_id'].nunique()}")
print(f"Names: psn_name {raw_sector['psn_name'].nunique()} -> applicants after method A + hand merges {links['applicant'].nunique()}")
print(f"Country: {(per_app['country_source'] == 'written').sum()} written, "
      f"{(per_app['country_source'] == 'office').sum()} from office, "
      f"{(per_app['country_source'] == 'hand').sum()} by hand, "
      f"{per_app['country'].isna().sum()} missing")

print("\nApplicants by type:")
print(per_app["type"].value_counts().to_string())

print("\nTop 10 applicants (families):")
print(links.groupby("applicant")["unit_id"].nunique().sort_values(ascending=False).head(10).to_string())

print("\nApplicants by country (top 5):")
print(per_app["country"].value_counts().head(5).to_string())

ENV: TEST  |  sector: Surface drones (unmanned surface vessels)

Sector families: 175 (codes 127, keywords 125, codes kept after drone words 50)
Families with applicants (2000-2025, PI): 153
Names: psn_name 163 -> applicants after method A + hand merges 159
Country: 63 written, 95 from office, 1 by hand, 0 missing

Applicants by type:
type
firm                       75
university                 42
individual                 28
government / non-profit    13
unknown                     1

Top 10 applicants (families):
applicant
SHANGHAI JIAO TONG UNIVERSITY                           5
HARBIN ENGINEERING UNIVERSITY                           4
SHANGHAI OCEAN UNIVERSITY                               3
KOREA INSTITUTE OF OCEAN SCIENCE TECHNOLOGY             3
NANJING UNIVERSITY OF INFORMATION SCIENCE TECHNOLOGY    3
SHANGHAI UNIVERSITY                                     3
ZHEJIANG OCEAN UNIVERSITY                               3
ZHEJIANG UNIVERSITY                                     3
DAL